# **Análise de Características**

In [2]:
!pip install -q pandas openpyxl scikit-learn joblib

In [3]:
import pandas as pd
import numpy as np
import joblib

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [4]:
uploaded = files.upload()

df = pd.read_excel("train_limpo.xlsx")

df["text"] = df["text"].fillna("").astype(str)
df = df.dropna(subset=["clarity"])

X = df["text"]
y = df["clarity"]

print("Total de registros:", len(df))
print("\nDistribuição das classes:")
print(y.value_counts())
print("\nProporção das classes:")
print(y.value_counts(normalize=True))

Saving train_limpo.xlsx to train_limpo.xlsx
Total de registros: 20091

Distribuição das classes:
clarity
c5      6892
c234    6853
c1      6346
Name: count, dtype: int64

Proporção das classes:
clarity
c5      0.343039
c234    0.341098
c1      0.315863
Name: proportion, dtype: float64


In [9]:
import numpy as np
import pandas as pd
import eli5

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

LABELS = ["c1", "c234", "c5"]

MIN_DF = 30
MIN_FOLDS = 4
MIN_PUREZA = 0.42
TOP = 12

textos = pd.Series(X).fillna("").astype(str).values

y = np.asarray(y)

if np.issubdtype(y.dtype, np.number):
    y = np.array([LABELS[int(v)] for v in y])

def novo_vec():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        sublinear_tf=True,
        max_features=60000
    )

def ajusta(textos, y):
    vec = novo_vec()
    X_tfidf = vec.fit_transform(textos)

    clf = LogisticRegression(
        C=4.0,
        max_iter=2000
    ).fit(X_tfidf, y)

    return vec, clf

grupos = pd.factorize(textos)[0]

gkf = GroupKFold(n_splits=5)

vocs = []

for fold, (idx_train, idx_val) in enumerate(
    gkf.split(textos, y, grupos), start=1
):
    vec_fold, clf_fold = ajusta(
        textos[idx_train],
        y[idx_train]
    )

    voc = np.array(vec_fold.get_feature_names_out())

    dominante = clf_fold.coef_.argmax(axis=0)

    vocs.append(
        pd.Series(dominante, index=voc, name=fold)
    )

    print(f"Fold {fold} concluído.")

D = pd.concat(vocs, axis=1)

vec, clf = ajusta(textos, y)

print("\nModelo final treinado.")

from IPython.display import display, HTML

explicacao = eli5.explain_weights(
    clf,
    vec=vec,
    top=30,
    target_names=LABELS
)

html = eli5.format_as_html(explicacao)

css = """
<style>
.eli5-weights,
.eli5-weights * {
    color: #000000 !important;
}

.eli5-weights th {
    color: #ffffff !important;
}
</style>
"""

print("\n========== TABELA ELI5 ==========")
display(HTML(css + html))

df_pesos = eli5.explain_weights_df(
    clf,
    vec=vec,
    top=400,
    target_names=LABELS
)

df_pesos = df_pesos[
    df_pesos["feature"] != "<BIAS>"
].rename(columns={
    "target": "classe",
    "feature": "termo",
    "weight": "peso"
})


X_binario = vec.transform(textos) > 0

vocabulario = np.array(vec.get_feature_names_out())

jmap = {
    termo: indice
    for indice, termo in enumerate(vocabulario)
}

linhas = []
armadilhas = []

for _, r in df_pesos.iterrows():

    termo = r["termo"]

    if termo not in jmap:
        continue

    classe = r["classe"]
    c = LABELS.index(classe)

    j = jmap[termo]

    n = int(X_binario[:, j].sum())

    if n < MIN_DF:
        continue

    if termo in D.index:
        estabilidade = int(
            (D.loc[termo] == c).sum()
        )
    else:
        estabilidade = 0

    indices = X_binario[:, j].nonzero()[0]

    distribuicao = (
        np.bincount(
            np.array([LABELS.index(v) for v in y[indices]]),
            minlength=3
        ) / n
    )

    item = {
        "classe": classe,
        "termo": termo,
        "peso": round(float(r["peso"]), 4),
        "quantidade": n,
        "estabilidade": estabilidade,
        "c1": distribuicao[0],
        "c234": distribuicao[1],
        "c5": distribuicao[2]
    }

    if (
        estabilidade >= MIN_FOLDS
        and distribuicao[c] >= MIN_PUREZA
    ):
        linhas.append(item)

    elif estabilidade >= MIN_FOLDS:
        armadilhas.append(item)

df_util = pd.DataFrame(linhas)
df_armadilhas = pd.DataFrame(armadilhas)

resultados = []

for classe in LABELS:

    parte = df_util[df_util["classe"] == classe].copy()

    parte["pureza"] = parte[classe]

    parte = parte.sort_values(
        "pureza",
        ascending=False
    ).head(TOP)

    resultados.append(parte)

if resultados and any(not p.empty for p in resultados):
    df_global = pd.concat(
        [p for p in resultados if not p.empty],
        ignore_index=True
    )
else:
    df_global = pd.DataFrame(columns=[
        "classe", "termo", "peso", "quantidade",
        "estabilidade", "c1", "c234", "c5"
    ])

if not df_armadilhas.empty:
    df_armadilhas = df_armadilhas.sort_values(
        "quantidade",
        ascending=False
    )

colunas = [
    "classe",
    "termo",
    "peso",
    "quantidade",
    "estabilidade",
    "c1",
    "c234",
    "c5"
]

print("\n========== TERMOS GLOBAIS ==========")

display(
    df_global[colunas].style.format({
        "peso": "{:.3f}",
        "c1": "{:.0%}",
        "c234": "{:.0%}",
        "c5": "{:.0%}"
    })
)

print("\n========== ARMADILHAS ==========")

if not df_armadilhas.empty:
    display(
        df_armadilhas[colunas].style.format({
            "peso": "{:.3f}",
            "c1": "{:.0%}",
            "c234": "{:.0%}",
            "c5": "{:.0%}"
        })
    )

df_global[colunas].to_excel(
    "interpretacao_global.xlsx",
    index=False
)

df_armadilhas[colunas].to_excel(
    "armadilhas.xlsx",
    index=False
)

print("\nArquivos salvos!")

Fold 1 concluído.
Fold 2 concluído.
Fold 3 concluído.
Fold 4 concluído.
Fold 5 concluído.

Modelo final treinado.

========== TABELA ELI5 ==========



========== TERMOS GLOBAIS ==========


,classe,termo,peso,quantidade,estabilidade,c1,c234,c5
0,c1,luis cesar,1.358,55,5,87%,11%,2%
1,c1,sr luis,2.178,72,5,86%,8%,6%
2,c1,pedido deve,1.652,32,5,75%,12%,12%
3,c1,respondidos,1.840,30,5,67%,13%,20%
4,c1,clareza,1.400,30,5,63%,23%,13%
5,c1,dados referente,1.207,47,5,62%,21%,17%
6,c1,favor acessar,1.533,39,5,62%,26%,13%
7,c1,possível identificar,1.360,36,5,61%,25%,14%
8,c1,furnas,1.292,34,5,59%,18%,24%
9,c1,luis,2.345,165,5,58%,23%,19%



========== ARMADILHAS ==========


,classe,termo,peso,quantidade,estabilidade,c1,c234,c5
49,c1,não,1.379,10234,5,33%,35%,31%
3,c1,informações,2.115,9547,5,35%,36%,29%
147,c234,as,1.199,7733,5,32%,38%,30%
154,c5,informamos,1.707,6976,5,31%,35%,34%
4,c1,solicitação,1.991,6272,5,34%,34%,32%
134,c234,dos,1.293,5955,5,31%,38%,30%
129,c234,sobre,1.335,4265,5,35%,38%,27%
183,c5,uma,1.414,3671,5,33%,38%,29%
9,c1,demanda,1.799,2565,5,36%,37%,27%
171,c5,endereço,1.487,2558,5,35%,35%,29%



Arquivos salvos!


In [10]:
from google.colab import files

uploaded = files.upload()

Saving teste1 - nosso.xlsx to teste1 - nosso.xlsx


In [11]:
from google.colab import files

uploaded = files.upload()

Saving teste1 - gabarito.xlsx to teste1 - gabarito.xlsx


In [12]:
import pandas as pd
nosso = pd.read_excel('teste1 - nosso.xlsx')
gabarito = pd.read_excel('teste1 - gabarito.xlsx')

In [13]:
df_comparacao = pd.merge(
    nosso[["resp_text", "clarity"]],
    gabarito[["resp_text", "clarity"]],
    on="resp_text",
    how="inner",
    suffixes=("_nosso", "_gabarito")
)

df_comparacao = df_comparacao.rename(columns={
    "clarity_nosso": "clarity_nosso",
    "clarity_gabarito": "clarity_gabarito"
})

display(df_comparacao.head())

,resp_text,clarity_nosso,clarity_gabarito
0,"Prezada Hellen, Dentre as pesquisas domici...",c234,c5
1,"Senhor Francisco, O Serviço de Informações a...",c234,c1
2,"Prezada Srª. Raylane, Em resposta a sua solic...",c5,c5
3,"Prezada Senhora, De acordo com o art.13 do D...",c234,c1
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",c1,c234


Palavra 'luis'

In [16]:
def destacar_celulas(df):
    estilos = pd.DataFrame(
        "",
        index=df.index,
        columns=df.columns
    )

    for i in df.index:
        for j in df.columns:

            if i == "Total" or j == "Total":
                estilos.loc[i, j] = "background-color: #ffffff"

            elif i == j:
                estilos.loc[i, j] = "background-color: #d4edda"

            else:
                estilos.loc[i, j] = "background-color: #f8d7da"

    return estilos

In [17]:
df_luis = df_comparacao[
    df_comparacao["resp_text"].str.contains(
        r"luis|sr luis",
        case=False,
        na=False,
        regex=True
    )
]

len(df_luis)

9

In [18]:
matriz = pd.crosstab(
    df_luis["clarity_nosso"],
    df_luis["clarity_gabarito"],
    margins=True,
    margins_name="Total"
)

display(matriz)

clarity_gabarito,c1,c234,c5,Total
clarity_nosso,,,,
c1,4,0,0,4
c234,0,1,1,2
c5,2,0,1,3
Total,6,1,2,9


Palavra 'solicitação'

In [19]:
df_solicitacao = df_comparacao[
    df_comparacao["resp_text"].str.contains(
        r"solicitação",
        case=False,
        na=False,
        regex=True
    )
]

len(df_solicitacao)

272

In [20]:
matriz_solicitacao = pd.crosstab(
    df_solicitacao["clarity_nosso"],
    df_solicitacao["clarity_gabarito"],
    margins=True,
    margins_name="Total"
)

In [21]:
matriz_solicitacao = matriz_solicitacao.rename_axis(
    index="EP1",
    columns="Gabarito"
)

display(
    matriz_solicitacao.style
    .apply(destacar_celulas, axis=None)
    .format("{:.0f}")
    .set_caption("Matriz de confusão — Ocorrências de 'solicitação''")
    .set_properties(**{
        "background-color": "#ffffff",
        "color": "#000000",
        "text-align": "center",
        "padding": "10px",
        "border": "1px solid #dee2e6"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#ffffff"),
                ("color", "#000000"),
                ("text-align", "center"),
                ("padding", "10px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("padding", "10px"),
                ("color", "#000000")
            ]
        }
    ])
)

Gabarito,c1,c234,c5,Total
EP1,,,,
c1,47,24,13,84
c234,29,37,13,79
c5,17,29,63,109
Total,93,90,89,272


In [ ]:
pd.set_option("display.max_colwidth", None)

display(df_solicitacao)

resp_text  \
1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         Senhor Francisco,  O Serviço de Informações ao Cidadão do Ministério do Planejamento, Desenvolvimento e Gestão agradece o seu contato.  Em atenção à sua solicitação, apresentamos, em anexo, a planilha solicitada esclarecendo que trata-se do período em que os pagamentos foram feitos via SIAPE, neste caso, de agosto a dezembro de 2016, conforme Art . 39 da Lei nº 13.327, de 29/07/2016.  Art. 39.  Para as competências de agosto a dezembro de 2016, os honorários advocatícios serão creditados em folha de pagamento pela União diretamente aos servidores ativos e aos aposentados nos cargos de que trata este Capítulo, no valor referente a uma cota-parte do montante 

In [ ]:
import re
import unicodedata

def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto if not unicodedata.combining(c)
    )
    return texto

padrao = re.compile(
    r"""
    \b(?:
        em\s+(?:resposta|atendimento)\s+a\s+sua\s+solicit\w*
        |
        em\s+atencao\s+a\s+(?:sua\s+)?solicit\w*
        |
        conforme\s+(?:a\s+)?solicit\w*
        |
        em\s+relacao\s+a\s+(?:sua\s+)?solicit\w*
        |
        a\s+resposta\s+a\s+sua\s+solicit\w*
        |
        sobre\s+(?:a\s+)?solicit\w*
        |
        em\s+resposta\s+a\s+vossa\s+solicit\w*
        |
        segue\s+(?:em\s+arquivo\s+)?(?:anexo\s+)?resposta\s+a\s+(?:sua\s+)?solicit\w*
        |
        segue\s+em\s+arquivo\s+anexo\s*,?\s*resposta\s+a\s+(?:sua\s+)?solicit\w*
        |
        respondemos\s+a\s+seguir\s+a\s+sua\s+solicit\w*
    )\b
    """,
    re.IGNORECASE | re.VERBOSE
)

df_solicitacao["resp_normalizado"] = (
    df_solicitacao["resp_text"].apply(normalizar_texto)
)

filtro = df_solicitacao["resp_normalizado"].str.contains(
    padrao,
    na=False
)

df_expressoes = df_solicitacao[filtro].copy()

print("Total de ocorrências:", len(df_expressoes))

Total de ocorrências: 142


/tmp/ipykernel_915/4098180870.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_solicitacao["resp_normalizado"] = (


In [ ]:
df_restantes = df_solicitacao[
    ~df_solicitacao["resp_text"].isin(df_expressoes["resp_text"])
].copy()

print("Total de casos restantes:", len(df_restantes))

display(df_restantes)

Total de casos restantes: 130


resp_text  \
8                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

Termo 'essa informação'

In [ ]:
df_info = df_comparacao[
    df_comparacao["resp_text"].str.contains(
        r"essa informação",
        case=False,
        na=False,
        regex=True
    )
]

len(df_info)

8

In [ ]:
matriz_info = pd.crosstab(
    df_info["clarity_nosso"],
    df_info["clarity_gabarito"],
    margins=True,
    margins_name="Total"
)

In [ ]:
matriz_info = matriz_info.rename_axis(
    index="EP1",
    columns="Gabarito"
)

display(
    matriz_info.style
    .apply(destacar_celulas, axis=None)
    .format("{:.0f}")
    .set_caption("Matriz de confusão — Ocorrências de 'solicitação''")
    .set_properties(**{
        "background-color": "#ffffff",
        "color": "#000000",
        "text-align": "center",
        "padding": "10px",
        "border": "1px solid #dee2e6"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#ffffff"),
                ("color", "#000000"),
                ("text-align", "center"),
                ("padding", "10px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("padding", "10px"),
                ("color", "#000000")
            ]
        }
    ])
)

Gabarito,c1,c234,c5,Total
EP1,,,,
c1,2,0,0,2
c234,1,3,1,5
c5,0,0,1,1
Total,3,3,2,8


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(df_info)

resp_text  \
38                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  Prezado(a) Senhor(a)     Informamos que o contrato comercial formalizado entre Correios e E-BAZAR, assim como com qualquer outro cliente, é um instrumento jurídico que gera direitos e obrigações entre contratante e contratado, razão pela qual tratam-se de informações restritas às partes envolvidas. Essa informação fundamenta-se nos artigos 5º e 6º, da Lei 6.538/78, onde as partes devem guardar sigilo absoluto sobre informações proprietárias e confidenciais necessárias à prestação dos serviços ora contratados, quais sejam, documentos, informações e programas inerentes aos serviços cont

Termo 'dnpm'

In [ ]:
df_dnpm = df_comparacao[
    df_comparacao["resp_text"].str.contains(
        r"dnpm",
        case=False,
        na=False,
        regex=True
    )
]

len(df_dnpm)

5

In [ ]:
matriz_dnpm = pd.crosstab(
    df_dnpm["clarity_nosso"],
    df_dnpm["clarity_gabarito"],
    margins=True,
    margins_name="Total"
)

In [ ]:
matriz_dnpm = matriz_dnpm.rename_axis(
    index="EP1",
    columns="Gabarito"
)

display(
    matriz_dnpm.style
    .apply(destacar_celulas, axis=None)
    .format("{:.0f}")
    .set_caption("Matriz de confusão — Ocorrências de 'solicitação''")
    .set_properties(**{
        "background-color": "#ffffff",
        "color": "#000000",
        "text-align": "center",
        "padding": "10px",
        "border": "1px solid #dee2e6"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#ffffff"),
                ("color", "#000000"),
                ("text-align", "center"),
                ("padding", "10px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("padding", "10px"),
                ("color", "#000000")
            ]
        }
    ])
)

Gabarito,c5,Total
EP1,,
c5,5,5
Total,5,5


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(df_dnpm)

,resp_text,clarity_nosso,clarity_gabarito
220,"Prezada Senhora Odila, Esterei em Brasília dia 25/08. Atenciosamente, SIC/DNPM",c5,c5
307,"Prezado Senhor Carlos, Informamos que não temos certidão negativa para comprovação de adimplência de nossas receitas, em face de não existir um controle automatizado para acompanhar as imposições de multas, atrasos no pagamento de taxas e de CFEM, principalmente de CFEM, visto ser um ato declaratório dos administrados, assim, só poderemos certificar que não existe dívida, se for fiscalizado. Atenciosamente, SIC/DNPM",c5,c5
326,"Prezado Senhor Carlos, Sobre o processo 850.062/2015: está em análise na Superintendência/PA, só podemos nos manifestar ao final, foi instaurado processo administrativo de nulidade mas ainda não está concluído. Sobre os processos 851.466/2013 e 851.465/2013: não cumpriu a exigências mas ainda não foi aberto processo administrativo de nulidade. Depois de aberto processo administrativo de nulidade abre-se prazo para defesa etc. Concluindo:só podemos dizer o que realmente ocorreu ao final da análise de cada processo. Atenciosamente, SIC/DNPM",c5,c5
685,"Prezada Senhora, Os e-mails que recebi com coordenadas forma esses abaixo e não consta nehum ponto com o que solicita. Atenciosamente, SIC/DNPM ----------------------------------------------------------------------------------------------- Dr. Paulo , Bom dia !!!! Estou enviando as coordenadas geográficas das Localidades Chapadinha e Baixão do são Pedro na região de Cacimba do Jatobá. área Chapadinha: 1. V1 - 09º 21' 12,220"" de Latitude Sul 43º 11' 45,630"" de Longitude Oeste 2. V2 - 09º 21' 12,220"" 43º 11' 22,639"" 3. V3 - 09º 21' 34,150"" 43º 11' 22,630"" 4. V4 - 09º 21' 34,150"" 43º 11' 45,630"" área Baixão do são Pedro: 1. V1 - 09º 19' 14,020"" de Latitude sul 43º 10' 21,310"" de Longitude Oeste 2. V2 - 09º 19' 14,020"" 43º 09' 56,880"" 3. V3 - 09º 19' 36,230"" 43º 10' 21,310"" 4. V4 - 09º 19' 36,230"" 43º 09' 56,880"" Obs: Qualquer dúvida ou recomendação pode me ligar pelo número: (86) 9 9911 3467 e 9 9918 3467 Sds. Valmir Batista -------------------------------------------------------------------------------- Dr. Paulo Esta é a coordenada da porteira da propriedade onde existe a Extração Mineral Ilegal nas proximidades da Cidade de Dirceu Arcoverde - Piauí. 09º 21' 24,165"" de Latitude Sul 42º 22' 26,712"" de Longitude Oeste Observação: Existe outra extração ilegal na mesma localidade, a Sra. Odila sabe a localização bem aproximada. Qualquer dúvida, estou a sua disposição. Telefone : (86) 9 9911 34 67 Valmir Batista",c5,c5
691,"Prezada Senhora Cristina, A servidora aposentada MARIA DAS GRAÇAS DOS SANTOS FERRO - matrícula SIAPE 453355, faleceu em 03/01/2017. Atenciosamente, SIC/DNPM",c5,c5


Termo 'ainda não'

In [ ]:
df_ainda_nao = df_comparacao[
    df_comparacao["resp_text"].str.contains(
        r"ainda não",
        case=False,
        na=False,
        regex=True
    )
]

len(df_ainda_nao)

13

In [ ]:
matriz_ainda_nao = pd.crosstab(
    df_ainda_nao["clarity_nosso"],
    df_ainda_nao["clarity_gabarito"],
    margins=True,
    margins_name="Total"
)

In [ ]:
matriz_ainda_nao = matriz_ainda_nao.rename_axis(
    index="EP1",
    columns="Gabarito"
)

display(
    matriz_ainda_nao.style
    .apply(destacar_celulas, axis=None)
    .format("{:.0f}")
    .set_caption("Matriz de confusão — Ocorrências de 'solicitação''")
    .set_properties(**{
        "background-color": "#ffffff",
        "color": "#000000",
        "text-align": "center",
        "padding": "10px",
        "border": "1px solid #dee2e6"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#ffffff"),
                ("color", "#000000"),
                ("text-align", "center"),
                ("padding", "10px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("padding", "10px"),
                ("color", "#000000")
            ]
        }
    ])
)

Gabarito,c234,c5,Total
EP1,,,
c234,3,1,4
c5,2,7,9
Total,5,8,13


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(df_ainda_nao)

,resp_text,clarity_nosso,clarity_gabarito
50,"Olá Nilse, seguem respostas: Nome: Gilvanete Azevedo Ferreira (http://libra.ifpi.edu.br/ouvidoria) Não há função gratificada Apenas a servidora e uma estagiária O Simplifique está disponível por meio do E-ouv apenas. Ainda não recebemos solicitação de simplificação Não possui Comitê de Desburocratização",c5,c5
67,"Prezado (a) Senhor (a), Comunicamos que seu pedido se refere à prestação de um serviço de atendimento ao contribuinte e não encontra respaldo na Lei de Acesso à Informação, portanto não é atendido pelo Serviço de Informação ao Cidadão. Para realizar comprovação da inscrição no CPF, orientamos que imprima comprovante de inscrição por meio do sítio da Receita Federal do Brasil (http://idg.receita.fazenda.gov.br/orientacao/tributaria/cadastros/cadastro-de-pessoas-fisicas-cpf/servicos/comprovante-de-inscricao-no-cpf). O número do CPF também pode ser encontrado em outros documentos, tais como: a) Carteira Nacional de Habilitação, b) Registro Civil de Nascimento, c) Carteira de Trabalho e Previdência Social (CTPS), d) Carteira de identidade profissional, e) Carteiras funcionais emitidas por órgãos públicos, f) Cartão magnético de movimentação de conta-corrente bancária, g) Talonário de cheque bancário, e h) Outros documentos de acesso a serviços de saúde pública, de assistência social ou a serviços previdenciários. Lembramos que as conveniadas não emitirão segunda via do cartão de CPF, conforme estabelece o § 1º do artigo 27 da Instrução Normativa RFB nº 1548, de 13 de fevereiro de 2015. Para mais informações sobre o assunto, sugerimos a leitura das questões 8, 34 e 39 do item Perguntas e respostas, disponível em: http://idg.receita.fazenda.gov.br/orientacao/tributaria/cadastros/cadastro-de-pessoas-fisicas-cpf/assuntos-relacionados/perguntas-e-respostas Outros esclarecimentos sobre CPF, informações sobre os órgãos fazendários e serviços prestados pelo Ministério da Fazenda (MF) poderão ser acessados pela Carta de Serviços do MF: http://fazenda.gov.br/carta-de-servicos/receita-federal-do-brasil/cadastro-de-pessoa-fisica-cpf Se, após seguir as orientações acima, ainda não obtiver as informações desejadas, procure uma unidade da RFB de sua jurisdição. Nesse caso, verifique com antecedência se é necessário agendar atendimento, acesse: https://idg.receita.fazenda.gov.br/contato/unidades-de-atendimento/unidades-de-atendimento Por fim, ressaltamos que o SIC lhe orienta a buscar as informações requeridas por meio dos canais adequados (acima indicados) por observar o disposto no artigo 11 da Lei nº 12.527/2011 e nos incisos IV e V do artigo 13 da Portaria MF nº 233/2012. Atenciosamente, Serviço de Informação ao Cidadão  SIC Ministério da Fazenda",c234,c234
84,"Prezado Senhor, Segue abaixo informação encaminhada pela Diretoria de Gestão de Pessoa: A ordem de convocação sairá no Edital de convocação de escolha de vaga, o qual ainda não foi divulgado tendo em vista que o MPOG não autorizou o provimento ainda. Atenciosamente, Sistema de Informação ao Cidadão  SIC",c234,c234
266,"Prezado Senhor, O Serviço de Informações ao Cidadão (SIC) da Agência Nacional de Energia Elétrica  ANEEL agradece o seu contato e informa que que os dados solicitados estão disponibilizados na planilha anexa, segregados por distribuidora. Salienta-se que as informações solicitadas foram obtidas da Base de Dados Geográfica da Distribuidora - BDGD com data base em 31/12/2016 e ainda não foram fiscalizadas pela ANEEL. Por fim, informamos que, conforme disposto no § Único do art. 15 da Lei nº 12527/11, há a possibilidade de interposição de recursos, por meio do sistema, no prazo de 10 (dez) dias. Atenciosamente, Superintendência de Regulação dos Serviços de Distribuição  SRD Serviço de Informação ao Cidadão  SIC Agência Nacional de Energia Elétrica  ANEEL",c234,c234
326,"Prezado Senhor Carlos, Sobre o processo 850.062/2015: está em análise na Superintendência/PA, só podemos nos manifestar ao final, foi instaurad

Termo 'olá'

In [ ]:
df_ola = df_comparacao[
    df_comparacao["resp_text"].str.contains(
        r"olá",
        case=False,
        na=False,
        regex=True
    )
]

len(df_ola)

6

In [ ]:
matriz_ola = pd.crosstab(
    df_ola["clarity_nosso"],
    df_ola["clarity_gabarito"],
    margins=True,
    margins_name="Total"
)

In [ ]:
matriz_ola = matriz_ola.rename_axis(
    index="EP1",
    columns="Gabarito"
)

display(
    matriz_ola.style
    .apply(destacar_celulas, axis=None)
    .format("{:.0f}")
    .set_caption("Matriz de confusão — Ocorrências de 'solicitação''")
    .set_properties(**{
        "background-color": "#ffffff",
        "color": "#000000",
        "text-align": "center",
        "padding": "10px",
        "border": "1px solid #dee2e6"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#ffffff"),
                ("color", "#000000"),
                ("text-align", "center"),
                ("padding", "10px"),
                ("border", "1px solid #dee2e6")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("padding", "10px"),
                ("color", "#000000")
            ]
        }
    ])
)

Gabarito,c1,c234,c5,Total
EP1,,,,
c1,0,0,1,1
c234,1,1,0,2
c5,0,0,3,3
Total,1,1,4,6


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(df_ola)

,resp_text,clarity_nosso,clarity_gabarito
26,"Olá, não há códigos de vaga em aberto para Professor de Educação Física no IFPI.",c5,c5
50,"Olá Nilse, seguem respostas: Nome: Gilvanete Azevedo Ferreira (http://libra.ifpi.edu.br/ouvidoria) Não há função gratificada Apenas a servidora e uma estagiária O Simplifique está disponível por meio do E-ouv apenas. Ainda não recebemos solicitação de simplificação Não possui Comitê de Desburocratização",c5,c5
201,"Olá Ellen, boa tarde, Encaminhamos abaixo resposta fornecida pela Coordenação de Administração de Pessoal e informamos que, de acordo com a Lei de Acesso à Informação (Lei n. 12.527/12) e o Decreto n. 7.724/12, há a possibilidade de recurso no prazo de 10 (dez) dias, nas formas e condições estabelecidas pelo artigo 15 e 17 da referida lei e decreto, respectivamente: ?? Em atenção a sua solicitação, cadastrada sob n. 23480009310201877, informamos que os códigos de vagas do IFMS são distribuídos conforme necessidade de admissão de servidores Técnicos-Administrativos nos Campi. O Campus Dourados tem 37 servidores em efetivo exercício em seu quadro. Conforme Portaria MEC nº 246 de 15 de abril de 2016, o Campus Dourados deve ter um quadro de 45 Técnicos-Administrativos. Agradecemos o contato. Atenciosamente, Serviço de Informação ao Cidadão (SIC) Instituto Federal de Educação, Ciência e Tecnologia de Mato Grosso do Sul.",c5,c5
290,"Prezada Senhora Adriana, Em atenção à solicitação de V.Sª, sobre dados relativos à postagem do Telegrama MZ557142565, cabe esclarecer que os Correios estão impedidos de fornecer os dados da transação comercial a terceiros não autorizados, do remetente e destinatário do objeto postal, conforme previsto nas legislações em vigor, a saber: 1. Constituição Federal de 1988: [...] Art. 5º Todos são iguais perante a lei, sem distinção de qualquer natureza, garantindo-se aos brasileiros e aos estrangeiros residentes no País a inviolabilidade do direito à vida, à liberdade, à igualdade, à segurança e à propriedade, nos termos seguintes: [...] XII - é inviolável o sigilo da correspondência e das comunicações telegráficas, de dados e das comunicações telefônicas, salvo, no último caso, por ordem judicial, nas hipóteses e na forma que a lei estabelecer para fins de investigação criminal ou instrução processual penal, 2. Lei nº 6.538, de 22 de junho de 1978: [...] Art. 5º - O sigilo da correspondência é inviolável. Parágrafo único - A ninguém é permitido intervir no serviço postal ou no serviço de telegrama, salvo nos casos e na forma previstos em lei. Art. 6º - As pessoas encarregadas do serviço postal ou do serviço de telegrama são obrigadas a manter segredo profissional sobre a existência de correspondência e do conteúdo de mensagem de que tenham conhecimento em razão de suas funções. Parágrafo único - Não se considera violação do segredo profissional, indispensável à manutenção do sigilo de correspondência a divulgação do nome do destinatário de objeto postal ou de telegrama que não tenha podido ser entregue por erro ou insuficiência de endereço. 3. Política de Privacidade do Serviço do CorreiosNet Shopping, que vincula os usuários do serviços com relação aos dados de cadastro, disponível no site (http://shopping.correios.com.br/wbm/shopping/script/politicaprivacidade.aspx): [...] 9. Privacidade dos Dados Cadastrais O CorreiosNet Shopping utiliza sistema de segurança apto a garantir a integridade e a confidencialidade do recolhimento, registro, utilização e transmissão eletrônica de dados pessoais. Os Correios agradecem a sua compreensão. HIRAN TEIXEIRA PARENTE Chefe de Departamento Eventuais recursos devem ser dirigidos ao Vice-Presidente do Negócio Postal , de acordo com o Art. 21 do Decreto 7.724/2012 que regulamenta a Lei de Acesso a Informação - Lei 12.527/2011, no prazo de 10 dias, a contar do recebimento desta resposta. Serviço de Informação ao Cidadão Empresa Brasileira de Correios e Telégrafos",c1,c5
434,"Olá, Sobre as vagas de técnicos em farmácia segue re